# FNO on Kuramoto-Sivashinsky (KS) — GPU Training
Runs the full FNO pipeline for the IEEE paper.  
**Runtime**: ~30-60 min on Colab T4 / Kaggle P100  
**Steps**: Single-step training → Curriculum rollout → Evaluation

In [ ]:
# Clone repo & install dependencies
!git clone https://github.com/Anthony50102/IEEE.git
%cd IEEE
!pip install -q neuraloperator h5netcdf xarray pyyaml

In [ ]:
# Fix config paths for Colab/Kaggle environment
import yaml, os

with open('fno/config/fno_ks_temporal_split_local.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['paths']['output_base'] = os.path.join(os.getcwd(), 'local_output/')
cfg['paths']['data_dir'] = os.path.join(os.getcwd(), 'data/ks')
os.makedirs(cfg['paths']['output_base'], exist_ok=True)

with open('fno/config/fno_ks_colab.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Config written. Data dir:", cfg['paths']['data_dir'])
print("Output base:", cfg['paths']['output_base'])

In [ ]:
# Verify GPU
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Step 1 — Single-step training
!python fno/step_1_train.py --config fno/config/fno_ks_colab.yaml --test

In [ ]:
# Find the run directory created by Step 1
import glob
run_dirs = sorted(glob.glob('local_output/fno_ks_temporal_split_*'))
RUN_DIR = run_dirs[-1]
print(f"Run directory: {RUN_DIR}")

In [ ]:
# Step 2 — Curriculum rollout training
!python fno/step_2_train.py --config fno/config/fno_ks_colab.yaml --run-dir {RUN_DIR} --test

In [ ]:
# Step 3 — Evaluation
!python fno/step_3_evaluate.py --config fno/config/fno_ks_colab.yaml --run-dir {RUN_DIR}

In [ ]:
# Display result figures
import matplotlib.pyplot as plt
from IPython.display import display, Image
import os

figures_dir = os.path.join(RUN_DIR, 'figures')
for fig_file in sorted(os.listdir(figures_dir)):
    if fig_file.endswith('.png'):
        print(f"\n--- {fig_file} ---")
        display(Image(os.path.join(figures_dir, fig_file)))

In [ ]:
# Zip results for download
import shutil
shutil.make_archive('fno_ks_results', 'zip', RUN_DIR)
print("Download fno_ks_results.zip from the file browser (left panel)")
# For Colab:
try:
    from google.colab import files
    files.download('fno_ks_results.zip')
except ImportError:
    print("Not running on Colab — download manually from file browser")

In [5]:
import os, shutil
src = "/kaggle/working/IEEE/local_output/fno_ks_temporal_split_20260317_174244"
dst = "/kaggle/working/fno_ks_slim"
os.makedirs(dst, exist_ok=True)

# Model checkpoint (needed to re-run eval)
shutil.copy2(os.path.join(src, "checkpoint_rollout_final.pt"), dst)

# Metrics & predictions (small)
for f in ["metrics.yaml", "predictions.npz", "status.txt",
         "step1_results.npz", "step2_results.npz"]:
   p = os.path.join(src, f)
   if os.path.exists(p):
       shutil.copy2(p, dst)

# All figures
fig_src = os.path.join(src, "figures")
if os.path.exists(fig_src):
   shutil.copytree(fig_src, os.path.join(dst, "figures"))

shutil.make_archive("fno_ks_slim", "zip", dst)
print(f"Done — download fno_ks_slim.zip ({os.path.getsize('/kaggle/working/fno_ks_slim.zip')/1e6:.1f} MB)")

Done — download fno_ks_slim.zip (97.2 MB)
